# Step 0: Explore listings.csv, choose columns, create the sample

Goal of this notebook (run it once, it is not part of the final deliverable):
1. Load the raw file and check that it is parsed correctly
2. Look at the schema and choose the columns we keep
3. Measure nulls and look at raw values, to fill in `data_contract.md`
4. Create a 5,000-row sample for fast work

Expected location: `notebooks/scratch/00_step0_explore.ipynb`, with the data in `data/raw/listings.csv`.

In [1]:
import os
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .master("local[*]")
         .appName("airbnb-step0")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())

DATA_PATH = "../../data/raw/listings.csv"
SAMPLE_DIR = "../../data/sample"
SAMPLE_SIZE = 5000
SEED = 42

assert os.path.exists(DATA_PATH), f"File not found: {os.path.abspath(DATA_PATH)}"

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/25 17:10:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 1. Load and check parsing
Descriptions contain line breaks and doubled quotes, so `multiLine` and `escape` are required.
Everything is read as text on purpose: we decide the types ourselves during cleaning.

In [2]:
raw = (spark.read
       .option("header", True)
       .option("multiLine", True)
       .option("escape", '"')
       .csv(DATA_PATH))

n_rows = raw.count()
print(f"Rows: {n_rows:,} | Columns: {len(raw.columns)}")

# If parsing broke, pieces of descriptions end up in the id column.
bad_ids = raw.filter(~F.col("id").rlike("^[0-9]+$")).count()
print("Rows with a non-numeric id (should be 0):", bad_ids)

Rows: 77,149 | Columns: 90


Rows with a non-numeric id (should be 0): 0


## 2. All columns

In [3]:
for i, c in enumerate(raw.columns):
    print(f"{i:3d}  {c}")

  0  id
  1  listing_url
  2  scrape_id
  3  last_scraped
  4  source
  5  name
  6  description
  7  neighborhood_overview
  8  picture_url
  9  host_id
 10  host_url
 11  host_profile_id
 12  host_profile_url
 13  host_name
 14  host_since
 15  hosts_time_as_user_years
 16  hosts_time_as_user_months
 17  hosts_time_as_host_years
 18  hosts_time_as_host_months
 19  host_location
 20  host_about
 21  host_response_time
 22  host_response_rate
 23  host_acceptance_rate
 24  host_is_superhost
 25  host_thumbnail_url
 26  host_picture_url
 27  host_neighbourhood
 28  host_listings_count
 29  host_total_listings_count
 30  host_verifications
 31  host_has_profile_pic
 32  host_identity_verified
 33  neighbourhood
 34  neighbourhood_cleansed
 35  neighbourhood_group_cleansed
 36  latitude
 37  longitude
 38  property_type
 39  room_type
 40  accommodates
 41  bathrooms
 42  bathrooms_text
 43  bedrooms
 44  beds
 45  amenities
 46  price
 47  price_quote_checkin_date
 48  price_quote_checko

## 3. Candidate columns
Edit this list after looking at the columns above. Missing ones are reported, not fatal.

In [4]:
CANDIDATES = [
    "id",
    "price",                                   # target
    "accommodates", "bedrooms", "beds", "bathrooms", "bathrooms_text",
    "room_type", "property_type",
    "neighbourhood_cleansed", "latitude", "longitude",
    "review_scores_rating", "number_of_reviews",
    "host_is_superhost", "host_response_rate",
    "minimum_nights", "availability_365", "instant_bookable",
    "amenities",
]

keep = [c for c in CANDIDATES if c in raw.columns]
missing = [c for c in CANDIDATES if c not in raw.columns]
print("Kept:", len(keep), "| Missing from this snapshot:", missing)

df = raw.select(keep).cache()
df.count()

Kept: 20 | Missing from this snapshot: []


77149

## 4. Null rate per column (%)
Empty strings count as null too.

In [5]:
null_pct = df.agg(*[
    F.round(100 * F.avg((F.col(c).isNull() | (F.trim(F.col(c)) == "")).cast("int")), 1).alias(c)
    for c in keep
]).toPandas().T.rename(columns={0: "null_%"}).sort_values("null_%", ascending=False)
print(null_pct.to_string())

26/11/25 17:10:31 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


                        null_%
host_response_rate       100.0
instant_bookable         100.0
bathrooms                100.0
price                     43.6
review_scores_rating      20.0
beds                       3.2
minimum_nights             0.2
bathrooms_text             0.1
bedrooms                   0.0
accommodates               0.0
id                         0.0
room_type                  0.0
longitude                  0.0
latitude                   0.0
neighbourhood_cleansed     0.0
property_type              0.0
host_is_superhost          0.0
number_of_reviews          0.0
availability_365           0.0
amenities                  0.0


## 5. Raw values of the tricky columns
Look at the formats to decide the cleaning rules.

In [6]:
tricky = [c for c in ["price", "bathrooms", "bathrooms_text", "host_response_rate",
                      "host_is_superhost", "instant_bookable", "review_scores_rating"] if c in keep]
df.select(tricky).show(15, truncate=False)

if "amenities" in keep:
    df.select("amenities").show(3, truncate=150)

+---------+---------+--------------+------------------+-----------------+----------------+--------------------+
|price    |bathrooms|bathrooms_text|host_response_rate|host_is_superhost|instant_bookable|review_scores_rating|
+---------+---------+--------------+------------------+-----------------+----------------+--------------------+
|$272.50  |NULL     |1 bath        |NULL              |f                |NULL            |4.9                 |
|$149.40  |NULL     |1 bath        |NULL              |f                |NULL            |4.72                |
|$177.75  |NULL     |1 bath        |NULL              |f                |NULL            |4.92                |
|$222.86  |NULL     |1 bath        |NULL              |f                |NULL            |4.92                |
|$57.53   |NULL     |1 bath        |NULL              |t                |NULL            |4.95                |
|$153.50  |NULL     |1 bath        |NULL              |f                |NULL            |4.85          

## 6. Quick look at the price (target)

In [7]:
priced = df.withColumn("price_num", F.regexp_replace("price", "[$,€\\s]", "").cast("double"))

n_price = priced.filter(F.col("price_num").isNotNull()).count()
print(f"Rows with a usable price: {n_price:,} of {n_rows:,} ({100 * n_price / n_rows:.1f}%)")

priced.select("price_num").summary("count", "min", "25%", "50%", "75%", "max").show()

q = priced.approxQuantile("price_num", [0.01, 0.05, 0.95, 0.99], 0.001)
print("Percentiles 1% / 5% / 95% / 99%:", q)

Rows with a usable price: 43,537 of 77,149 (56.4%)


+-------+---------+
|summary|price_num|
+-------+---------+
|  count|    43537|
|    min|    10.38|
|    25%|    139.0|
|    50%|    218.0|
|    75%|    359.0|
|    max| 100666.1|
+-------+---------+



Percentiles 1% / 5% / 95% / 99%: [46.28, 70.0, 849.55, 1786.67]


## 7. Categorical columns: how many values?

In [8]:
for c in ["room_type", "neighbourhood_cleansed", "property_type"]:
    if c in keep:
        print(f"\n{c}: {df.select(c).distinct().count()} distinct values")
        df.groupBy(c).count().orderBy(F.desc("count")).show(25, truncate=False)


room_type: 4 distinct values


+---------------+-----+
|room_type      |count|
+---------------+-----+
|Entire home/apt|68333|
|Private room   |8280 |
|Hotel room     |386  |
|Shared room    |150  |
+---------------+-----+


neighbourhood_cleansed: 20 distinct values


+----------------------+-----+
|neighbourhood_cleansed|count|
+----------------------+-----+
|Buttes-Montmartre     |8064 |
|Popincourt            |6898 |
|Vaugirard             |6163 |
|Entrepôt              |5291 |
|Batignolles-Monceau   |5075 |
|Passy                 |4700 |
|Buttes-Chaumont       |4195 |
|Ménilmontant          |4075 |
|Opéra                 |3712 |
|Temple                |3474 |
|Reuilly               |3131 |
|Bourse                |2868 |
|Observatoire          |2748 |
|Élysée                |2604 |
|Panthéon              |2589 |
|Hôtel-de-Ville        |2501 |
|Gobelins              |2499 |
|Luxembourg            |2405 |
|Palais-Bourbon        |2255 |
|Louvre                |1902 |
+----------------------+-----+




property_type: 60 distinct values


+---------------------------------+-----+
|property_type                    |count|
+---------------------------------+-----+
|Entire rental unit               |64222|
|Private room in rental unit      |4986 |
|Entire condo                     |2057 |
|Room in hotel                    |1801 |
|Entire loft                      |832  |
|Private room in bed and breakfast|580  |
|Room in boutique hotel           |565  |
|Entire home                      |488  |
|Private room in condo            |289  |
|Entire serviced apartment        |245  |
|Entire townhouse                 |233  |
|Private room in home             |111  |
|Shared room in rental unit       |89   |
|Room in aparthotel               |86   |
|Private room in townhouse        |69   |
|Private room in loft             |59   |
|Private room in hostel           |45   |
|Entire place                     |43   |
|Shared room in hotel             |34   |
|Private room                     |32   |
|Private room in guest suite      

## 8. Create the 5,000-row sample
Only the kept columns, still raw (not cleaned), so the same cleaning code works on both files.
Only rows **with a price** are sampled: rows without the target are dropped by cleaning anyway
(43.6% of the raw file), so keeping them would waste almost half of the sample.
Written with pandas to avoid Spark's winutils problem on Windows.

In [9]:
priced_raw = df.filter(F.col("price").isNotNull() & (F.trim(F.col("price")) != ""))
n_priced = priced_raw.count()

fraction = min(1.0, SAMPLE_SIZE * 1.2 / n_priced)
sample_pdf = priced_raw.sample(fraction=fraction, seed=SEED).limit(SAMPLE_SIZE).toPandas()

os.makedirs(SAMPLE_DIR, exist_ok=True)
sample_path = os.path.join(SAMPLE_DIR, "listings_sample.csv")
sample_pdf.to_csv(sample_path, index=False)
print(f"Sample written: {sample_path} ({len(sample_pdf):,} rows)")

# Check it reads back with the same options as the full file
check = (spark.read.option("header", True).option("multiLine", True)
         .option("escape", '"').csv(sample_path))
print("Re-read:", check.count(), "rows,", len(check.columns), "columns")
assert check.count() == SAMPLE_SIZE

Sample written: ../../data/sample/listings_sample.csv (5,000 rows)


Re-read: 5000 rows, 20 columns
